#  Learning Unsupervised Embeddings for Molecules

In this tutorial, we will use a `SeqToSeq` model to generate fingerprints for classifying molecules.  This is based on the following paper, although some of the implementation details are different: Xu et al., "Seq2seq Fingerprint: An Unsupervised Deep Molecular Embedding for Drug Discovery" (https://doi.org/10.1145/3107411.3107424).

## Colab

This tutorial and the rest in this sequence can be done in Google colab. If you'd like to open this notebook in colab, you can use the following link.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/deepchem/deepchem/blob/master/examples/tutorials/Learning_Unsupervised_Embeddings_for_Molecules.ipynb)



In [2]:
# !pip install --pre deepchem
import deepchem
deepchem.__version__

'2.5.0'

In [3]:
import numpy as np
import time

import deepchem as dc
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ExponentialLR

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


# Learning Embeddings with SeqToSeq

Many types of models require their inputs to have a fixed shape.  Since molecules can vary widely in the numbers of atoms and bonds they contain, this makes it hard to apply those models to them.  We need a way of generating a fixed length "fingerprint" for each molecule.  Various ways of doing this have been designed, such as the Extended-Connectivity Fingerprints (ECFPs) we used in earlier tutorials.  But in this example, instead of designing a fingerprint by hand, we will let a `SeqToSeq` model learn its own method of creating fingerprints.

A `SeqToSeq` model performs sequence to sequence translation.  For example, they are often used to translate text from one language to another.  It consists of two parts called the "encoder" and "decoder".  The encoder is a stack of recurrent layers.  The input sequence is fed into it, one token at a time, and it generates a fixed length vector called the "embedding vector".  The decoder is another stack of recurrent layers that performs the inverse operation: it takes the embedding vector as input, and generates the output sequence.  By training it on appropriately chosen input/output pairs, you can create a model that performs many sorts of transformations.

In this case, we will use SMILES strings describing molecules as the input sequences.  We will train the model as an autoencoder, so it tries to make the output sequences identical to the input sequences.  For that to work, the encoder must create embedding vectors that contain all information from the original sequence.  That's exactly what we want in a fingerprint, so perhaps those embedding vectors will then be useful as a way to represent molecules in other models!

Let's start by loading the data.  We will use the MUV dataset.  It includes 74,501 molecules in the training set, and 9313 molecules in the validation set, so it gives us plenty of SMILES strings to work with.

In [5]:
# Load dataset using DeepChem
tasks, datasets, transformers = dc.molnet.load_muv(splitter='stratified')
train_dataset, valid_dataset, test_dataset = datasets
train_smiles = train_dataset.ids
valid_smiles = valid_dataset.ids

[10:01:14] DEPRECATION WARNING: please use MorganGenerator
[10:01:14] DEPRECATION WARNING: please use MorganGenerator
[10:01:14] DEPRECATION WARNING: please use MorganGenerator
[10:01:14] DEPRECATION WARNING: please use MorganGenerator
[10:01:14] DEPRECATION WARNING: please use MorganGenerator
[10:01:14] DEPRECATION WARNING: please use MorganGenerator
[10:01:14] DEPRECATION WARNING: please use MorganGenerator
[10:01:14] DEPRECATION WARNING: please use MorganGenerator
[10:01:14] DEPRECATION WARNING: please use MorganGenerator
[10:01:14] DEPRECATION WARNING: please use MorganGenerator
[10:01:14] DEPRECATION WARNING: please use MorganGenerator
[10:01:14] DEPRECATION WARNING: please use MorganGenerator
[10:01:14] DEPRECATION WARNING: please use MorganGenerator
[10:01:14] DEPRECATION WARNING: please use MorganGenerator
[10:01:14] DEPRECATION WARNING: please use MorganGenerator
[10:01:14] DEPRECATION WARNING: please use MorganGenerator
[10:01:14] DEPRECATION WARNING: please use MorganGenerat

We need to define the "alphabet" for our `SeqToSeq` model, the list of all tokens that can appear in sequences.  (It's also possible for input and output sequences to have different alphabets, but since we're training it as an autoencoder, they're identical in this case.)  Make a list of every character that appears in any training sequence.

In [6]:
# Extract tokens form the SMILES strings
tokens = set()
for s in train_smiles:
    tokens = tokens.union(set(c for c in s))

# Add special tokens (for PyTorch implementation)
tokens.add('<START>')
tokens.add('<END>')

tokens = sorted(list(tokens))

In [7]:
# For PyTorch, create a mapping from tokens to indices and vice versa to prepare for DataLoader
token_to_idx = {token: idx for idx, token in enumerate(tokens)}
idx_to_token = {idx: token for token, idx in token_to_idx.items()}

In [8]:
# Define PyTorch Dataset and dataloader
class SMILESDataset(Dataset):
    def __init__(self, smiles_list, token_to_idx, max_length):
        self.smiles_list = smiles_list
        self.token_to_idx = token_to_idx
        self.max_length = max_length

    def __len__(self):
        return len(self.smiles_list)

    def __getitem__(self, idx):
        smiles = self.smiles_list[idx]
        # Convert SMILES string to a sequence of token indices
        input_seq = [self.token_to_idx[token] for token in smiles]
        # Add padding to the sequence to make it `max_length`
        input_seq = input_seq + [0] * (self.max_length - len(input_seq))
        input_seq = torch.tensor(input_seq, dtype=torch.long)

        # For simplicity, use the same sequence as the target
        target_seq = input_seq.clone()
        return input_seq, target_seq

# Max sequence length
max_length = max(len(s) for s in train_smiles)

# Create the training and valid dataset and DataLoader
batch_size = 100 
train_dataset = SMILESDataset(train_smiles, token_to_idx, max_length)
valid_dataset = SMILESDataset(valid_smiles, token_to_idx, max_length)
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
valid_dataloader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False)

Create the model and define the optimization method to use.  In this case, learning works much better if we gradually decrease the learning rate.  We use an `ExponentialDecay` to multiply the learning rate by 0.9 after each epoch.

In [9]:
import inspect
import deepchem.models

# print(inspect.getfile(deepchem.models.SeqToSeq))
print(deepchem.models.SeqToSeq.__bases__)
print(deepchem.models.__file__)

(<class 'deepchem.models.keras_model.KerasModel'>,)
/home/jantine/miniconda3/envs/deepchem/lib/python3.12/site-packages/deepchem/models/__init__.py


In [ ]:
# class VariationalRandomizer(Layer):
#     """Add random noise to the embedding and include a corresponding loss."""

#     def __init__(self, embedding_dimension, annealing_start_step,
#                  annealing_final_step, **kwargs):
#         super(VariationalRandomizer, self).__init__(**kwargs)
#         self._embedding_dimension = embedding_dimension
#         self._annealing_final_step = annealing_final_step
#         self._annealing_start_step = annealing_start_step
#         self.dense_mean = Dense(embedding_dimension)
#         self.dense_stddev = Dense(embedding_dimension)
#         self.combine = layers.CombineMeanStd(training_only=True)

#     def call(self, inputs, training=True):
#         input, global_step = inputs
#         embedding_mean = self.dense_mean(input)
#         embedding_stddev = self.dense_stddev(input)
#         embedding = self.combine([embedding_mean, embedding_stddev],
#                                  training=training)
#         mean_sq = embedding_mean * embedding_mean
#         stddev_sq = embedding_stddev * embedding_stddev
#         kl = mean_sq + stddev_sq - tf.math.log(stddev_sq + 1e-20) - 1
#         anneal_steps = self._annealing_final_step - self._annealing_start_step
#         if anneal_steps > 0:
#             current_step = tf.cast(global_step,
#                                    tf.float32) - self._annealing_start_step
#             anneal_frac = tf.maximum(0.0, current_step) / anneal_steps
#             kl_scale = tf.minimum(1.0, anneal_frac * anneal_frac)
#         else:
#             kl_scale = 1.0
#         self.add_loss(0.5 * kl_scale * tf.reduce_mean(kl))
#         return embedding

In [10]:
# TensorFlow
class SeqToSeq(KerasModel):
  """Implements sequence to sequence translation models.

  The model is based on the description in Sutskever et al., "Sequence to
  Sequence Learning with Neural Networks" (https://arxiv.org/abs/1409.3215),
  although this implementation uses GRUs instead of LSTMs.  The goal is to
  take sequences of tokens as input, and translate each one into a different
  output sequence.  The input and output sequences can both be of variable
  length, and an output sequence need not have the same length as the input
  sequence it was generated from.  For example, these models were originally
  developed for use in natural language processing.  In that context, the
  input might be a sequence of English words, and the output might be a
  sequence of French words.  The goal would be to train the model to translate
  sentences from English to French.

  The model consists of two parts called the "encoder" and "decoder".  Each one
  consists of a stack of recurrent layers.  The job of the encoder is to
  transform the input sequence into a single, fixed length vector called the
  "embedding".  That vector contains all relevant information from the input
  sequence.  The decoder then transforms the embedding vector into the output
  sequence.

  These models can be used for various purposes.  First and most obviously,
  they can be used for sequence to sequence translation.  In any case where you
  have sequences of tokens, and you want to translate each one into a different
  sequence, a SeqToSeq model can be trained to perform the translation.

  Another possible use case is transforming variable length sequences into
  fixed length vectors.  Many types of models require their inputs to have a
  fixed shape, which makes it difficult to use them with variable sized inputs
  (for example, when the input is a molecule, and different molecules have
  different numbers of atoms).  In that case, you can train a SeqToSeq model as
  an autoencoder, so that it tries to make the output sequence identical to the
  input one.  That forces the embedding vector to contain all information from
  the original sequence.  You can then use the encoder for transforming
  sequences into fixed length embedding vectors, suitable to use as inputs to
  other types of models.

  Another use case is to train the decoder for use as a generative model.  Here
  again you begin by training the SeqToSeq model as an autoencoder.  Once
  training is complete, you can supply arbitrary embedding vectors, and
  transform each one into an output sequence.  When used in this way, you
  typically train it as a variational autoencoder.  This adds random noise to
  the encoder, and also adds a constraint term to the loss that forces the
  embedding vector to have a unit Gaussian distribution.  You can then pick
  random vectors from a Gaussian distribution, and the output sequences should
  follow the same distribution as the training data.

  When training as a variational autoencoder, it is best to use KL cost
  annealing, as described in https://arxiv.org/abs/1511.06349.  The constraint
  term in the loss is initially set to 0, so the optimizer just tries to
  minimize the reconstruction loss.  Once it has made reasonable progress
  toward that, the constraint term can be gradually turned back on.  The range
  of steps over which this happens is configurable.
  """

  sequence_end = object()

  def __init__(self,
               input_tokens,
               output_tokens,
               max_output_length,
               encoder_layers=4,
               decoder_layers=4,
               embedding_dimension=512,
               dropout=0.0,
               reverse_input=True,
               variational=False,
               annealing_start_step=5000,
               annealing_final_step=10000,
               **kwargs):
    """Construct a SeqToSeq model.

    In addition to the following arguments, this class also accepts all the keyword arguments
    from TensorGraph.

    Parameters
    ----------
    input_tokens: list
      a list of all tokens that may appear in input sequences
    output_tokens: list
      a list of all tokens that may appear in output sequences
    max_output_length: int
      the maximum length of output sequence that may be generated
    encoder_layers: int
      the number of recurrent layers in the encoder
    decoder_layers: int
      the number of recurrent layers in the decoder
    embedding_dimension: int
      the width of the embedding vector.  This also is the width of all
      recurrent layers.
    dropout: float
      the dropout probability to use during training
    reverse_input: bool
      if True, reverse the order of input sequences before sending them into
      the encoder.  This can improve performance when working with long sequences.
    variational: bool
      if True, train the model as a variational autoencoder.  This adds random
      noise to the encoder, and also constrains the embedding to follow a unit
      Gaussian distribution.
    annealing_start_step: int
      the step (that is, batch) at which to begin turning on the constraint term
      for KL cost annealing
    annealing_final_step: int
      the step (that is, batch) at which to finish turning on the constraint term
      for KL cost annealing
    """
    if SeqToSeq.sequence_end not in input_tokens:
      input_tokens = input_tokens + [SeqToSeq.sequence_end]
    if SeqToSeq.sequence_end not in output_tokens:
      output_tokens = output_tokens + [SeqToSeq.sequence_end]
    self._input_tokens = input_tokens
    self._output_tokens = output_tokens
    self._input_dict = dict((x, i) for i, x in enumerate(input_tokens))
    self._output_dict = dict((x, i) for i, x in enumerate(output_tokens))
    self._max_output_length = max_output_length
    self._embedding_dimension = embedding_dimension
    self._reverse_input = reverse_input
    self.encoder = self._create_encoder(encoder_layers, dropout)
    self.decoder = self._create_decoder(decoder_layers, dropout)
    
    features = self._create_features()
    gather_indices = Input(shape=(2,), dtype=tf.int32)
    global_step = Input(shape=tuple(), dtype=tf.int32)
    embedding = self.encoder([features, gather_indices])
    self._embedding = self.encoder([features, gather_indices], training=False)
    if variational:
      randomizer = VariationalRandomizer(
          self._embedding_dimension, annealing_start_step, annealing_final_step)
      embedding = randomizer([self._embedding, global_step])
      self._embedding = randomizer(
          [self._embedding, global_step], training=False)
    output = self.decoder(embedding)
    model = tf.keras.Model(
        inputs=[features, gather_indices, global_step], outputs=output)
    super(SeqToSeq, self).__init__(model, self._create_loss(), **kwargs)

  def _create_features(self):
    return Input(shape=(None, len(self._input_tokens)))

  def _create_encoder(self, n_layers, dropout):
    """Create the encoder as a tf.keras.Model."""
    input = self._create_features()
    gather_indices = Input(shape=(2,), dtype=tf.int32)
    prev_layer = input
    for i in range(n_layers):
      if dropout > 0.0:
        prev_layer = Dropout(rate=dropout)(prev_layer)
      prev_layer = GRU(
          self._embedding_dimension, return_sequences=True)(prev_layer)
    prev_layer = Lambda(lambda x: tf.gather_nd(x[0], x[1]))(
        [prev_layer, gather_indices])
    return tf.keras.Model(inputs=[input, gather_indices], outputs=prev_layer)

  def _create_decoder(self, n_layers, dropout):
    """Create the decoder as a tf.keras.Model."""
    input = Input(shape=(self._embedding_dimension,))
    prev_layer = layers.Stack()(self._max_output_length * [input])
    for i in range(n_layers):
      if dropout > 0.0:
        prev_layer = Dropout(dropout)(prev_layer)
      prev_layer = GRU(
          self._embedding_dimension, return_sequences=True)(prev_layer)
    output = Dense(
        len(self._output_tokens), activation=tf.nn.softmax)(prev_layer)
    return tf.keras.Model(inputs=input, outputs=output)

  def _create_loss(self):
    """Create the loss function."""

    def loss_fn(outputs, labels, weights):
      prob = tf.reduce_sum(outputs[0] * labels[0], axis=2)
      mask = tf.reduce_sum(labels[0], axis=2)
      log_prob = tf.math.log(prob + 1e-20) * mask
      loss = -tf.reduce_mean(tf.reduce_sum(log_prob, axis=1))
      return loss + sum(self.model.losses)

    return loss_fn

  def fit_sequences(self,
                    sequences,
                    max_checkpoints_to_keep=5,
                    checkpoint_interval=1000,
                    restore=False):
    """Train this model on a set of sequences

    Parameters
    ----------
    sequences: iterable
      the training samples to fit to.  Each sample should be
      represented as a tuple of the form (input_sequence, output_sequence).
    max_checkpoints_to_keep: int
      the maximum number of checkpoints to keep.  Older checkpoints are discarded.
    checkpoint_interval: int
      the frequency at which to write checkpoints, measured in training steps.
    restore: bool
      if True, restore the model from the most recent checkpoint and continue training
      from there.  If False, retrain the model from scratch.
    """
    self.fit_generator(
        self._generate_batches(sequences),
        max_checkpoints_to_keep=max_checkpoints_to_keep,
        checkpoint_interval=checkpoint_interval,
        restore=restore)

  def predict_from_sequences(self, sequences, beam_width=5):
    """Given a set of input sequences, predict the output sequences.

    The prediction is done using a beam search with length normalization.

    Parameters
    ----------
    sequences: iterable
      the input sequences to generate a prediction for
    beam_width: int
      the beam width to use for searching.  Set to 1 to use a simple greedy search.
    """
    result = []
    for batch in self._batch_elements(sequences):
      features = self._create_input_array(batch)
      indices = np.array([(i, len(batch[i]) if i < len(batch) else 0)
                          for i in range(self.batch_size)])
      probs = self.predict_on_generator([[(features, indices,
                                           np.array(self.get_global_step())),
                                          None, None]])
      for i in range(len(batch)):
        result.append(self._beam_search(probs[i], beam_width))
    return result

  def predict_from_embeddings(self, embeddings, beam_width=5):
    """Given a set of embedding vectors, predict the output sequences.

    The prediction is done using a beam search with length normalization.

    Parameters
    ----------
    embeddings: iterable
      the embedding vectors to generate predictions for
    beam_width: int
      the beam width to use for searching.  Set to 1 to use a simple greedy search.
    """
    result = []
    for batch in self._batch_elements(embeddings):
      embedding_array = np.zeros(
          (self.batch_size, self._embedding_dimension), dtype=np.float32)
      for i, e in enumerate(batch):
        embedding_array[i] = e
      probs = self.decoder(embedding_array, training=False)
      probs = probs.numpy()
      for i in range(len(batch)):
        result.append(self._beam_search(probs[i], beam_width))
    return result

  def predict_embeddings(self, sequences):
    """Given a set of input sequences, compute the embedding vectors.

    Parameters
    ----------
    sequences: iterable
      the input sequences to generate an embedding vector for
    """
    result = []
    for batch in self._batch_elements(sequences):
      features = self._create_input_array(batch)
      indices = np.array([(i, len(batch[i]) if i < len(batch) else 0)
                          for i in range(self.batch_size)])
      embeddings = self.predict_on_generator(
          [[(features, indices, np.array(self.get_global_step())), None, None]],
          outputs=self._embedding)
      for i in range(len(batch)):
        result.append(embeddings[i])
    return np.array(result, dtype=np.float32)

  def _beam_search(self, probs, beam_width):
    """Perform a beam search for the most likely output sequence."""
    if beam_width == 1:
      # Do a simple greedy search.

      s = []
      for i in range(len(probs)):
        token = self._output_tokens[np.argmax(probs[i])]
        if token == SeqToSeq.sequence_end:
          break
        s.append(token)
      return s

    # Do a beam search with length normalization.

    logprobs = np.log(probs)
    # Represent each candidate as (normalized prob, raw prob, sequence)
    candidates = [(0.0, 0.0, [])]
    for i in range(len(logprobs)):
      new_candidates = []
      for c in candidates:
        if len(c[2]) > 0 and c[2][-1] == SeqToSeq.sequence_end:
          # This candidate sequence has already been terminated
          if len(new_candidates) < beam_width:
            heappush(new_candidates, c)
          else:
            heappushpop(new_candidates, c)
        else:
          # Consider all possible tokens we could add to this candidate sequence.
          for j, logprob in enumerate(logprobs[i]):
            new_logprob = logprob + c[1]
            newc = (new_logprob / (len(c[2]) + 1), new_logprob,
                    c[2] + [self._output_tokens[j]])
            if len(new_candidates) < beam_width:
              heappush(new_candidates, newc)
            else:
              heappushpop(new_candidates, newc)
      candidates = new_candidates
    return sorted(candidates)[-1][2][:-1]

  def _create_input_array(self, sequences):
    """Create the array describing the input sequences for a batch."""
    lengths = [len(x) for x in sequences]
    if self._reverse_input:
      sequences = [reversed(s) for s in sequences]
    features = np.zeros(
        (self.batch_size, max(lengths) + 1, len(self._input_tokens)),
        dtype=np.float32)
    for i, sequence in enumerate(sequences):
      for j, token in enumerate(sequence):
        features[i, j, self._input_dict[token]] = 1
    features[np.arange(len(sequences)), lengths, self._input_dict[
        SeqToSeq.sequence_end]] = 1
    return features

  def _create_output_array(self, sequences):
    """Create the array describing the target sequences for a batch."""
    lengths = [len(x) for x in sequences]
    labels = np.zeros(
        (self.batch_size, self._max_output_length, len(self._output_tokens)),
        dtype=np.float32)
    end_marker_index = self._output_dict[SeqToSeq.sequence_end]
    for i, sequence in enumerate(sequences):
      for j, token in enumerate(sequence):
        labels[i, j, self._output_dict[token]] = 1
      for j in range(lengths[i], self._max_output_length):
        labels[i, j, end_marker_index] = 1
    return labels

  def _batch_elements(self, elements):
    """Combine elements into batches."""
    batch = []
    for s in elements:
      batch.append(s)
      if len(batch) == self.batch_size:
        yield batch
        batch = []
    if len(batch) > 0:
      yield batch

  def _generate_batches(self, sequences):
    """Create feed_dicts for fitting."""
    for batch in self._batch_elements(sequences):
      inputs = []
      outputs = []
      for input, output in batch:
        inputs.append(input)
        outputs.append(output)
      for i in range(len(inputs), self.batch_size):
        inputs.append([])
        outputs.append([])
      features = self._create_input_array(inputs)
      labels = self._create_output_array(outputs)
      gather_indices = np.array([(i, len(x)) for i, x in enumerate(inputs)])
      yield ([features, gather_indices,
              np.array(self.get_global_step())], [labels], [])

NameError: name 'KerasModel' is not defined

In [14]:
import deepchem.models
print(deepchem.models.SeqToSeq.__bases__)
# print(deepchem.__file__)

(<class 'deepchem.models.keras_model.KerasModel'>,)


In [15]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class VariationalRandomizer(nn.Module):
    """Add random noise to the embedding and include a corresponding loss."""

    def __init__(self, embedding_dimension, annealing_start_step, annealing_final_step):
        super(VariationalRandomizer, self).__init__()
        self._embedding_dimension = embedding_dimension
        self._annealing_start_step = annealing_start_step
        self._annealing_final_step = annealing_final_step

        # Define dense layers for mean and standard deviation
        self.dense_mean = nn.Linear(embedding_dimension, embedding_dimension)
        self.dense_stddev = nn.Linear(embedding_dimension, embedding_dimension)

        # self.combine is not necessary in PyTorch, an is implemented using tensor operations in forward

    def forward(self, inputs, global_step, training=True):
        """
        Parameters:
        - inputs: Tensor of shape (batch_size, embedding_dimension)
        - global_step: Scalar tensor indicating the current training step
        - training: Boolean indicating whether the model is in training mode

        Returns:
        - Processed embedding with variational noise applied
        """
        input_tensor = inputs

        # Compute mean and standard deviation
        embedding_mean = self.dense_mean(input_tensor)
        embedding_stddev = self.dense_stddev(input_tensor)

        # Combine mean and stddev to create the embedding
        if training:
            # Reparameterization trick: z = mean + stddev * epsilon
            epsilon = torch.randn_like(embedding_stddev)
            embedding = embedding_mean + torch.exp(0.5 * embedding_stddev) * epsilon
        else:
            # In evaluation mode, use the mean as the embedding
            embedding = embedding_mean

        # Compute KL divergence
        mean_sq = embedding_mean ** 2
        # stddev_sq = torch.exp(embedding_stddev) ** 2
        stddev_sq = embedding_stddev ** 2  # No exp here, as stddev is directly σ
        kl = mean_sq + stddev_sq - torch.log(stddev_sq + 1e-20) - 1

        # Compute annealing factor
        anneal_steps = self._annealing_final_step - self._annealing_start_step
        if anneal_steps > 0:
            current_step = global_step - self._annealing_start_step
            anneal_frac = torch.clamp(current_step / anneal_steps, min=0.0, max=1.0)
            kl_scale = anneal_frac ** 2
        else:
            kl_scale = 1.0

        # Add KL divergence loss
        kl_loss = 0.5 * kl_scale * torch.mean(kl)
        if training:
            self.add_loss(kl_loss)  # Custom loss handling may be needed

        return embedding

    def add_loss(self, loss):
        """
        Add the loss to the module. PyTorch does not have a built-in `add_loss` method like Keras,
        so this one is handled manually in the training loop.
        """
        if not hasattr(self, 'losses'):
            self.losses = []
        self.losses.append(loss)

In [ ]:
# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from heapq import heappush, heappushpop

class SeqToSeq(nn.Module):
  """Implements sequence to sequence translation models.
  
  (Full original docstring retained)
  """

  sequence_end = object()

  def __init__(self,
               input_tokens,
               output_tokens,
               max_output_length,
               encoder_layers=4,
               decoder_layers=4,
               embedding_dimension=512,
               dropout=0.0,
               reverse_input=True,
               variational=False,
               annealing_start_step=5000,
               annealing_final_step=10000,
               **kwargs):
    """Construct a SeqToSeq model.

    In addition to the following arguments, this class also accepts all the keyword arguments
    from TensorGraph.

    Parameters
    ----------
    input_tokens: list
      a list of all tokens that may appear in input sequences
    output_tokens: list
      a list of all tokens that may appear in output sequences
    max_output_length: int
      the maximum length of output sequence that may be generated
    encoder_layers: int
      the number of recurrent layers in the encoder
    decoder_layers: int
      the number of recurrent layers in the decoder
    embedding_dimension: int
      the width of the embedding vector.  This also is the width of all
      recurrent layers.
    dropout: float
      the dropout probability to use during training
    reverse_input: bool
      if True, reverse the order of input sequences before sending them into
      the encoder.  This can improve performance when working with long sequences.
    variational: bool
      if True, train the model as a variational autoencoder.  This adds random
      noise to the encoder, and also constrains the embedding to follow a unit
      Gaussian distribution.
    annealing_start_step: int
      the step (that is, batch) at which to begin turning on the constraint term
      for KL cost annealing
    annealing_final_step: int
      the step (that is, batch) at which to finish turning on the constraint term
      for KL cost annealing
    """
    super(SeqToSeq, self).__init__()

    if SeqToSeq.sequence_end not in input_tokens:
      input_tokens = input_tokens + [SeqToSeq.sequence_end]
    if SeqToSeq.sequence_end not in output_tokens:
      output_tokens = output_tokens + [SeqToSeq.sequence_end]

    self._input_tokens = input_tokens
    self._output_tokens = output_tokens
    self._input_dict = {x: i for i, x in enumerate(input_tokens)}
    self._output_dict = {x: i for i, x in enumerate(output_tokens)}
    self._max_output_length = max_output_length
    self._embedding_dimension = embedding_dimension
    self._reverse_input = reverse_input
    
    self.encoder = self._create_encoder(encoder_layers, dropout)
    self.decoder = self._create_decoder(decoder_layers, dropout)

    features = self._create_features()

    # These are directly handled in the forward in PyTorch
    # gather_indices = Input(shape=(2,), dtype=tf.int32)
    # global_step = Input(shape=tuple(), dtype=tf.int32)
    # embedding = self.encoder([features, gather_indices])
    # self._embedding = self.encoder([features, gather_indices], training=False)
    
    self.variational = variational



    # Variables needed for KL annealing
    self.annealing_start_step = annealing_start_step
    self.annealing_final_step = annealing_final_step
    self.global_step = 0

  def _create_encoder(self, n_layers, dropout):
    layers = []
    for _ in range(n_layers):
      layers.append(nn.GRU(
          input_size=len(self._input_tokens),
          hidden_size=self._embedding_dimension,
          batch_first=True))
      if dropout > 0.0:
        layers.append(nn.Dropout(dropout))
    return nn.ModuleList(layers)

  def _create_decoder(self, n_layers, dropout):
    layers = []
    for _ in range(n_layers):
      layers.append(nn.GRU(
          input_size=self._embedding_dimension,
          hidden_size=self._embedding_dimension,
          batch_first=True))
      if dropout > 0.0:
        layers.append(nn.Dropout(dropout))
    layers.append(nn.Linear(self._embedding_dimension, len(self._output_tokens)))
    return nn.ModuleList(layers)

  def encode(self, features, gather_indices):
    x = features
    for layer in self.encoder:
      if isinstance(layer, nn.GRU):
        x, _ = layer(x)
      else:
        x = layer(x)
    batch_indices = gather_indices[:, 0]
    time_indices = gather_indices[:, 1]
    gathered = x[batch_indices, time_indices]
    return gathered

  def decode(self, embedding):
    repeated = embedding.unsqueeze(1).repeat(1, self._max_output_length, 1)
    x = repeated
    for layer in self.decoder[:-1]:
      if isinstance(layer, nn.GRU):
        x, _ = layer(x)
      else:
        x = layer(x)
    output = self.decoder[-1](x)
    output = F.softmax(output, dim=2)
    return output

  def forward(self, features, gather_indices):
    embedding = self.encode(features, gather_indices)
    if self.variational:
      embedding = self.apply_variational(embedding)
    output = self.decode(embedding)
    return output

  def apply_variational(self, embedding):
    # Very simple variational noise addition (needs elaboration for full VAE)
    if self.training:
      stddev = torch.ones_like(embedding)
      noise = torch.randn_like(embedding) * stddev
      embedding = embedding + noise
    return embedding

  def loss_fn(self, outputs, labels, weights=None):
    prob = torch.sum(outputs * labels, dim=2)
    mask = torch.sum(labels, dim=2)
    log_prob = torch.log(prob + 1e-20) * mask
    loss = -torch.mean(torch.sum(log_prob, dim=1))
    return loss

  def fit_sequences(self, sequences, optimizer, epochs=1):
    """Train this model on a set of sequences (simple loop version)."""
    self.train()
    for epoch in range(epochs):
      for batch in self._batch_elements(sequences):
        inputs, outputs = zip(*batch)
        features = self._create_input_array(inputs)
        labels = self._create_output_array(outputs)
        gather_indices = torch.tensor([(i, len(seq)) for i, seq in enumerate(inputs)], dtype=torch.long)
        
        optimizer.zero_grad()
        predictions = self.forward(features, gather_indices)
        loss = self.loss_fn(predictions, labels)
        loss.backward()
        optimizer.step()
        self.global_step += 1

  def predict_from_sequences(self, sequences, beam_width=5):
    """Given a set of input sequences, predict the output sequences."""
    self.eval()
    result = []
    with torch.no_grad():
      for batch in self._batch_elements(sequences):
        features = self._create_input_array(batch)
        gather_indices = torch.tensor([(i, len(batch[i])) for i in range(len(batch))], dtype=torch.long)
        probs = self.forward(features, gather_indices).cpu().numpy()
        for i in range(len(batch)):
          result.append(self._beam_search(probs[i], beam_width))
    return result

  def predict_from_embeddings(self, embeddings, beam_width=5):
    """Given a set of embedding vectors, predict the output sequences."""
    self.eval()
    result = []
    with torch.no_grad():
      for batch in self._batch_elements(embeddings):
        embedding_array = torch.tensor(batch, dtype=torch.float32)
        probs = self.decode(embedding_array).cpu().numpy()
        for i in range(len(batch)):
          result.append(self._beam_search(probs[i], beam_width))
    return result

  def predict_embeddings(self, sequences):
    """Given a set of input sequences, compute the embedding vectors."""
    self.eval()
    result = []
    with torch.no_grad():
      for batch in self._batch_elements(sequences):
        features = self._create_input_array(batch)
        gather_indices = torch.tensor([(i, len(batch[i])) for i in range(len(batch))], dtype=torch.long)
        embeddings = self.encode(features, gather_indices).cpu().numpy()
        result.append(embeddings)
    return np.concatenate(result, axis=0)

  def _beam_search(self, probs, beam_width):
    """Perform a beam search for the most likely output sequence."""
    if beam_width == 1:
      # Simple greedy search
      s = []
      for i in range(len(probs)):
        token = self._output_tokens[np.argmax(probs[i])]
        if token == SeqToSeq.sequence_end:
          break
        s.append(token)
      return s

    logprobs = np.log(probs)
    candidates = [(0.0, 0.0, [])]
    for i in range(len(logprobs)):
      new_candidates = []
      for c in candidates:
        if len(c[2]) > 0 and c[2][-1] == SeqToSeq.sequence_end:
          if len(new_candidates) < beam_width:
            heappush(new_candidates, c)
          else:
            heappushpop(new_candidates, c)
        else:
          for j, logprob in enumerate(logprobs[i]):
            new_logprob = logprob + c[1]
            newc = (new_logprob / (len(c[2]) + 1), new_logprob, c[2] + [self._output_tokens[j]])
            if len(new_candidates) < beam_width:
              heappush(new_candidates, newc)
            else:
              heappushpop(new_candidates, newc)
      candidates = new_candidates
    return sorted(candidates)[-1][2][:-1]

  def _create_input_array(self, sequences):
    lengths = [len(x) for x in sequences]
    if self._reverse_input:
      sequences = [list(reversed(s)) for s in sequences]
    features = torch.zeros((len(sequences), max(lengths)+1, len(self._input_tokens)), dtype=torch.float32)
    for i, sequence in enumerate(sequences):
      for j, token in enumerate(sequence):
        features[i, j, self._input_dict[token]] = 1
      features[i, lengths[i], self._input_dict[SeqToSeq.sequence_end]] = 1
    return features

  def _create_output_array(self, sequences):
    lengths = [len(x) for x in sequences]
    labels = torch.zeros((len(sequences), self._max_output_length, len(self._output_tokens)), dtype=torch.float32)
    end_marker_index = self._output_dict[SeqToSeq.sequence_end]
    for i, sequence in enumerate(sequences):
      for j, token in enumerate(sequence):
        labels[i, j, self._output_dict[token]] = 1
      for j in range(lengths[i], self._max_output_length):
        labels[i, j, end_marker_index] = 1
    return labels

  def _batch_elements(self, elements, batch_size=32):
    """Combine elements into batches."""
    batch = []
    for s in elements:
      batch.append(s)
      if len(batch) == batch_size:
        yield batch
        batch = []
    if len(batch) > 0:
      yield batch


In [ ]:
# class SeqToSeq(nn.Module):
#     """Implements sequence-to-sequence translation models in PyTorch."""

#     sequence_end = "<END>"

#     def __init__(self,
#                  input_tokens,
#                  output_tokens,
#                  max_output_length,
#                  encoder_layers=4,
#                  decoder_layers=4,
#                  embedding_dimension=512,
#                  dropout=0.0,
#                  reverse_input=True,
#                  variational=False,
#                  annealing_start_step=5000,
#                  annealing_final_step=10000,
#                  **kwargs):
#         """
#         Construct a SeqToSeq model.

#         Parameters
#         ----------
#         input_tokens: list
#             List of all tokens that may appear in input sequences.
#         output_tokens: list
#             List of all tokens that may appear in output sequences.
#         max_output_length: int
#             Maximum length of output sequence that may be generated.
#         encoder_layers: int
#             Number of recurrent layers in the encoder.
#         decoder_layers: int
#             Number of recurrent layers in the decoder.
#         embedding_dimension: int
#             Dimension of the embedding vector and GRU hidden states.
#         dropout: float
#             Dropout probability during training.
#         reverse_input: bool
#             If True, reverse the input sequence before encoding.
#         variational: bool
#             If True, train the model as a variational autoencoder.
#         annealing_start_step: int
#             Step at which to begin turning on the constraint term for KL cost annealing.
#         annealing_final_step: int
#             Step at which to finish turning on the constraint term for KL cost annealing.
#         """
#         super(SeqToSeq, self).__init__()
        
#         self._input_tokens = input_tokens
#         self._output_tokens = output_tokens
#         self._input_dict = {token: idx for idx, token in enumerate(input_tokens)}
#         self._output_dict = {token: idx for idx, token in enumerate(output_tokens)}
#         self._max_output_length = max_output_length  # Initialize max_output_length
#         self._embedding_dimension = embedding_dimension
#         self._reverse_input = reverse_input
        
        
#         self.variational = variational
#         self.annealing_start_step = annealing_start_step
#         self.annealing_final_step = annealing_final_step

#         # Embedding layers
#         self.input_embedding = nn.Embedding(len(input_tokens), embedding_dimension)
#         self.output_embedding = nn.Embedding(len(output_tokens), embedding_dimension)

#         # Encoder
#         self.encoder = nn.GRU(embedding_dimension, embedding_dimension,
#                               num_layers=encoder_layers, batch_first=True, dropout=dropout)

#         # Decoder
#         self.decoder = nn.GRU(embedding_dimension, embedding_dimension,
#                               num_layers=decoder_layers, batch_first=True, dropout=dropout)
#         self.output_layer = nn.Linear(embedding_dimension, len(output_tokens))

#         # Variational components
#         if variational:
#             self.mean_layer = nn.Linear(embedding_dimension, embedding_dimension)
#             self.std_layer = nn.Linear(embedding_dimension, embedding_dimension)

#     def _create_features(self):
#         """This function is redundant in PyTorch."""
#         # In PyTorch, we directly use tensors as inputs, so this function is unnecessary.
#         pass

#     def _create_encoder(self, n_layers, dropout):
#         """This function is redundant in PyTorch."""
#         # The encoder is already defined in the `__init__` method using `nn.GRU`.
#         pass

#     def _create_decoder(self, n_layers, dropout):
#         """This function is redundant in PyTorch."""
#         # The decoder is already defined in the `__init__` method using `nn.GRU` and `nn.Linear`.
#         pass

#     def _create_loss(self):
#         """This function is redundant in PyTorch."""
#         # In PyTorch, we use built-in loss functions like `nn.CrossEntropyLoss` directly.
#         pass

#     def forward(self, input_seq, target_seq=None, global_step=None):
#         """
#         Forward pass for training or inference.

#         Parameters
#         ----------
#         input_seq: torch.Tensor
#             Input sequence tensor of shape (batch_size, seq_length).
#         target_seq: torch.Tensor, optional
#             Target sequence tensor of shape (batch_size, seq_length). If None, inference is performed.
#         global_step: int, optional
#             Current training step for KL cost annealing (used in variational autoencoder).

#         Returns
#         -------
#         output: torch.Tensor
#             Output sequence tensor of shape (batch_size, max_output_length, vocab_size).
#         """
#         # Reverse input sequence if required
#         if self.reverse_input:
#             input_seq = torch.flip(input_seq, dims=[1])

#         # Encode input sequence
#         embedded_input = self.input_embedding(input_seq)
#         _, hidden = self.encoder(embedded_input)

#         # Variational autoencoder
#         if self.variational:
#             mean = self.mean_layer(hidden[-1])
#             std = torch.exp(0.5 * self.std_layer(hidden[-1]))
#             z = mean + std * torch.randn_like(std)

#             # KL cost annealing
#             if global_step is not None:
#                 anneal_steps = self.annealing_final_step - self.annealing_start_step
#                 if anneal_steps > 0:
#                     current_step = max(0, global_step - self.annealing_start_step)
#                     kl_scale = min(1.0, (current_step / anneal_steps) ** 2)
#                 else:
#                     kl_scale = 1.0
#                 kl_loss = 0.5 * kl_scale * torch.mean(mean ** 2 + std ** 2 - torch.log(std ** 2 + 1e-20) - 1)
#                 self.add_loss(kl_loss)
#             hidden = z.unsqueeze(0).repeat(self.encoder.num_layers, 1, 1)

#         # Decode output sequence
#         if target_seq is not None:
#             # Training mode
#             embedded_target = self.output_embedding(target_seq)
#             decoder_output, _ = self.decoder(embedded_target, hidden)
#         else:
#             # Inference mode
#             batch_size = input_seq.size(0)
#             decoder_input = torch.full((batch_size, 1), self.output_dict['<START>'],
#                                     dtype=torch.long, device=input_seq.device)
#             decoder_output = []
#             for _ in range(self.max_output_length):
#                 embedded_decoder_input = self.output_embedding(decoder_input)
#                 output, hidden = self.decoder(embedded_decoder_input, hidden)
#                 output_token = torch.argmax(self.output_layer(output), dim=-1)
#                 decoder_output.append(output_token)
#                 decoder_input = output_token
#             decoder_output = torch.cat(decoder_output, dim=1)

#             # Embed the decoder_output tokens
#             decoder_output = self.output_embedding(decoder_output)

#         # Reshape decoder_output for the Linear layer
#         decoder_output = decoder_output.reshape(-1, self.embedding_dimension)  # Flatten the sequence dimension

#         # Compute output probabilities
#         output = self.output_layer(decoder_output.float())  # Convert to float
#         output = output.view(input_seq.size(0), self.max_output_length, -1)  # Reshape back to (batch_size, seq_length, vocab_size)
#         return output

#     def fit_sequences(self, dataloader, optimizer, scheduler, device, epochs=1):
#         """
#         Train the model on a set of sequences.

#         Parameters
#         ----------
#         dataloader: DataLoader
#             DataLoader for the training dataset.
#         optimizer: torch.optim.Optimizer
#             Optimizer for training.
#         scheduler: torch.optim.lr_scheduler._LRScheduler
#             Learning rate scheduler.
#         device: torch.device
#             Device to train the model on (CPU or GPU).
#         epochs: int
#             Number of epochs to train for.
#         """
#         self.train()
#         for epoch in range(epochs):
#             total_loss = 0
#             for batch in dataloader:
#                 input_seq, target_seq = batch
#                 input_seq, target_seq = input_seq.to(device), target_seq.to(device)

#                 optimizer.zero_grad()
#                 output = self(input_seq, target_seq)
#                 loss = F.cross_entropy(output.view(-1, output.size(-1)), target_seq.view(-1), ignore_index=0)
#                 loss.backward()
#                 optimizer.step()
#                 total_loss += loss.item()
#             scheduler.step()
#             print(f"Epoch {epoch + 1}/{epochs}, Loss: {total_loss / len(dataloader)}")

#     def predict_from_sequences(self, sequences, token_to_idx, idx_to_token, device):
#         """
#         Predict output sequences for a list of input sequences.

#         Parameters
#         ----------
#         sequences: list
#             List of input sequences (e.g., SMILES strings).
#         token_to_idx: dict
#             Mapping from tokens to indices.
#         idx_to_token: dict
#             Mapping from indices to tokens.
#         device: torch.device
#             Device to run the model on (CPU or GPU).

#         Returns
#         -------
#         list
#             List of predicted sequences.
#         """
#         self.eval()
#         predicted_sequences = []
#         with torch.no_grad():
#             for seq in sequences:
#                 # Tokenize the input sequence
#                 input_seq = [token_to_idx[token] for token in seq]
#                 input_seq = input_seq + [0] * (self.max_output_length - len(input_seq))
#                 input_seq = torch.tensor(input_seq, dtype=torch.long).unsqueeze(0).to(device)
                
#                 # Predict the output sequence
#                 output = self(input_seq)
#                 predicted_tokens = torch.argmax(output, dim=-1).squeeze(0).tolist()
                
#                 # Convert token indices back to SMILES string
#                 predicted_sequences.append(''.join([idx_to_token[idx] for idx in predicted_tokens if idx != 0]))
#         return predicted_sequences

#     def predict_from_embeddings(self, embeddings, device):
#         """
#         Predict output sequences from embedding vectors.

#         Parameters
#         ----------
#         embeddings: torch.Tensor
#             Embedding vectors of shape (batch_size, embedding_dimension).
#         device: torch.device
#             Device to run the model on (CPU or GPU).

#         Returns
#         -------
#         list
#             List of predicted sequences.
#         """
#         self.eval()
#         predicted_sequences = []
#         with torch.no_grad():
#             for embedding in embeddings:
#                 embedding = embedding.unsqueeze(0).to(device)
#                 hidden = embedding.repeat(self.decoder.num_layers, 1, 1)
#                 decoder_input = torch.full((1, 1), self.output_dict['<START>'], dtype=torch.long, device=device)
#                 output_sequence = []
#                 for _ in range(self.max_output_length):
#                     embedded_decoder_input = self.output_embedding(decoder_input)
#                     output, hidden = self.decoder(embedded_decoder_input, hidden)
#                     output_token = torch.argmax(self.output_layer(output), dim=-1)
#                     output_sequence.append(output_token.item())
#                     decoder_input = output_token
#                 predicted_sequences.append(''.join([self.output_tokens[idx] for idx in output_sequence if idx != 0]))
#         return predicted_sequences

#     def predict_embeddings(self, sequences):
#         """
#         Compute embedding vectors for a set of input sequences.

#         Parameters
#         ----------
#         sequences: list
#             List of input sequences (e.g., SMILES strings).

#         Returns
#         -------
#         np.ndarray
#             Embedding vectors of shape (num_samples, embedding_dimension).
#         """
#         self.eval()
#         embeddings = []
#         with torch.no_grad():
#             for seq in sequences:
#                 # Tokenize the input sequence
#                 input_seq = [self.input_dict[token] for token in seq]
#                 input_seq = input_seq + [0] * (self.max_output_length - len(input_seq))  # Use self.max_output_length
#                 input_seq = torch.tensor(input_seq, dtype=torch.long).unsqueeze(0).to(next(self.parameters()).device)

#                 # Compute the embedding
#                 embedded_input = self.input_embedding(input_seq)
#                 _, hidden = self.encoder(embedded_input)
#                 embeddings.append(hidden[-1].cpu().numpy())  # Convert to NumPy array
#         return np.array(embeddings, dtype=np.float32)
    
#     def predict_embeddings_batch(self, sequences, batch_size=32):
#         """
#         Compute embedding vectors for a set of input sequences in batches.

#         Parameters
#         ----------
#         sequences: list
#             List of input sequences (e.g., SMILES strings).
#         batch_size: int
#             Number of sequences to process in each batch.

#         Returns
#         -------
#         np.ndarray
#             Embedding vectors of shape (num_samples, embedding_dimension).
#         """
#         self.eval()
#         embeddings = []
#         with torch.no_grad():
#             for batch_start in range(0, len(sequences), batch_size):
#                 batch = sequences[batch_start:batch_start + batch_size]
#                 input_seqs = []
#                 for seq in batch:
#                     input_seq = [self.input_dict[token] for token in seq]
#                     input_seq = input_seq + [0] * (self.max_output_length - len(input_seq))
#                     input_seqs.append(input_seq)
#                 input_seqs = torch.tensor(input_seqs, dtype=torch.long).to(next(self.parameters()).device)

#                 # Compute the embeddings
#                 embedded_input = self.input_embedding(input_seqs)
#                 _, hidden = self.encoder(embedded_input)
#                 embeddings.extend(hidden[-1].cpu().numpy())  # Convert to NumPy array
#         return np.array(embeddings, dtype=np.float32)
    

#     def _beam_search(self, probs, beam_width):
#         """
#         Perform a beam search for the most likely output sequence.

#         Parameters
#         ----------
#         probs: torch.Tensor
#             Probabilities of shape (seq_length, vocab_size).
#         beam_width: int
#             Beam width for searching.

#         Returns
#         -------
#         list
#             Most likely output sequence.
#         """
#         if beam_width == 1:
#             # Greedy search
#             return [self.output_tokens[torch.argmax(p).item()] for p in probs]

#         # Beam search
#         logprobs = torch.log(probs)
#         candidates = [(0.0, [])]
#         for step_probs in logprobs:
#             new_candidates = []
#             for score, seq in candidates:
#                 for idx, logprob in enumerate(step_probs):
#                     new_candidates.append((score + logprob.item(), seq + [idx]))
#             candidates = sorted(new_candidates, key=lambda x: x[0], reverse=True)[:beam_width]
#         return [self.output_tokens[idx] for idx in candidates[0][1]]

#     def _create_input_array(self, sequences):
#         """This function is redundant in PyTorch."""
#         # In PyTorch, we directly use tensors for input sequences.
#         pass

#     def _create_output_array(self, sequences):
#         """This function is redundant in PyTorch."""
#         # In PyTorch, we directly use tensors for target sequences.
#         pass

#     def _batch_elements(self, elements, batch_size):
#         """
#         Combine elements into batches.

#         Parameters
#         ----------
#         elements: list
#             List of elements to batch.
#         batch_size: int
#             Batch size.

#         Yields
#         ------
#         list
#             Batches of elements.
#         """
#         for i in range(0, len(elements), batch_size):
#             yield elements[i:i + batch_size]

#     def _generate_batches(self, sequences, batch_size):
#         """
#         Generate batches of input/output pairs for training.

#         Parameters
#         ----------
#         sequences: list
#             List of input/output sequence pairs.
#         batch_size: int
#             Batch size.

#         Yields
#         ------
#         tuple
#             Batches of input and output sequences.
#         """
#         for batch in self._batch_elements(sequences, batch_size):
#             inputs, outputs = zip(*batch)
#             yield torch.tensor(inputs, dtype=torch.long), torch.tensor(outputs, dtype=torch.long)

In [ ]:
# Define the model
model = SeqToSeq(
    input_tokens=tokens,
    output_tokens=tokens,
    max_output_length=max_length,
    encoder_layers=2,
    decoder_layers=2,
    embedding_dimension=256,
    dropout=0.0,
    reverse_input=True,
    variational=False
)

# Move the model to the appropriate device (GPU if available, otherwise CPU)
model.to(device)

SeqToSeq(
  (input_embedding): Embedding(31, 256)
  (output_embedding): Embedding(31, 256)
  (encoder): GRU(256, 256, num_layers=2, batch_first=True)
  (decoder): GRU(256, 256, num_layers=2, batch_first=True)
  (output_layer): Linear(in_features=256, out_features=31, bias=True)
)

Let's train it!  The input to `fit_sequences()` is a generator that produces input/output pairs.  On a good GPU, this should take a few hours or less.

In [ ]:
# def generate_sequences(epochs):
#   for i in range(epochs):
#     for s in train_smiles:
#       yield (s, s)

# model.fit_sequences(generate_sequences(40))

In [ ]:
# Define optimizer and learning rate scheduler
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.9)

# Train the model for 40 epochs using the DataLoader
# epochs = 40
epochs = 3
model.fit_sequences(train_dataloader, optimizer, scheduler, device, epochs=epochs)

Epoch 1/3, Loss: 0.04459700137536954
Epoch 2/3, Loss: 0.00012032122122891084
Epoch 3/3, Loss: 4.696299339480145e-05


Let's see how well it works as an autoencoder.  We'll run the first 500 molecules from the validation set through it, and see how many of them are exactly reproduced.

In [ ]:
# predicted = model.predict_from_sequences(valid_smiles[:500])
# count = 0
# for s,p in zip(valid_smiles[:500], predicted):
#   if ''.join(p) == s:
#     count += 1
# print('reproduced', count, 'of 500 validation SMILES strings')

In [ ]:
# Predict sequences for the first 500 validation SMILES strings
predicted = model.predict_from_sequences(valid_smiles[:500], token_to_idx, idx_to_token, device)

# Count how many SMILES strings were reproduced exactly
count = 0
for s, p in zip(valid_smiles[:500], predicted):
    if ''.join(p) == s:
        count += 1
print('reproduced', count, 'of 500 validation SMILES strings')

reproduced 0 of 500 validation SMILES strings


Now we'll trying using the encoder as a way to generate molecular fingerprints.  We compute the embedding vectors for all molecules in the training and validation datasets, and create new datasets that have those as their feature vectors.  The amount of data is small enough that we can just store everything in memory.

In [ ]:
# Start timing
time_start = time.time()

# Compute embeddings for the training dataset
train_embeddings = model.predict_embeddings(train_smiles)
train_embeddings_dataset = dc.data.NumpyDataset(
    X=train_embeddings,
    y=datasets[0].y,
    w=datasets[0].w.astype(np.float32),
    ids=datasets[0].ids
)

# Compute embeddings for the validation dataset
valid_embeddings = model.predict_embeddings(valid_smiles)
valid_embeddings_dataset = dc.data.NumpyDataset(
    X=valid_embeddings,
    y=datasets[1].y,
    w=datasets[1].w.astype(np.float32),
    ids=datasets[1].ids
)

# End timing
time_end = time.time()

# Print the elapsed time
print(f"Time taken for embedding computation and dataset creation: {time_end - time_start:.2f} seconds")

Time taken for embedding computation and dataset creation: 123.18 seconds


In [ ]:
# Use batch processing for efficiency
time_start = time.time()

# Compute embeddings for the training dataset
train_embeddings = model.predict_embeddings_batch(train_smiles, batch_size=32)
train_embeddings_dataset = dc.data.NumpyDataset(
    X=train_embeddings,
    y=datasets[0].y,  # Labels from the original DeepChem dataset
    w=datasets[0].w.astype(np.float32),  # Weights from the original DeepChem dataset
    ids=datasets[0].ids  # IDs from the original DeepChem dataset
)

# Compute embeddings for the validation dataset
valid_embeddings = model.predict_embeddings_batch(valid_smiles, batch_size=32)
valid_embeddings_dataset = dc.data.NumpyDataset(
    X=valid_embeddings,
    y=datasets[1].y,  # Labels from the original DeepChem dataset
    w=datasets[1].w.astype(np.float32),  # Weights from the original DeepChem dataset
    ids=datasets[1].ids  # IDs from the original DeepChem dataset
)

time_end = time.time()
print(f"Time taken for embedding computation and dataset creation: {time_end - time_start:.2f} seconds")

Time taken for embedding computation and dataset creation: 6.44 seconds


In [ ]:
# import torch
# import torch.nn as nn
# import torch.nn.functional as F
# from torch.utils.data import DataLoader
# from typing import List, Sequence, Union, Tuple, Iterable

# OneOrMany = Union[float, List[float]]

# class MultitaskClassifier(nn.Module):
#     def __init__(self,
#                  n_tasks: int,
#                  n_features: int,
#                  layer_sizes: Sequence[int] = [1000],
#                  weight_init_stddevs: OneOrMany = 0.02,
#                  bias_init_consts: OneOrMany = 1.0,
#                  weight_decay_penalty: float = 0.0,
#                  weight_decay_penalty_type: str = "l2",
#                  dropouts: OneOrMany = 0.5,
#                  activation_fns: OneOrMany = nn.ReLU,
#                  n_classes: int = 2,
#                  residual: bool = False,
#                  **kwargs) -> None:
#         super(MultitaskClassifier, self).__init__()

#         self.n_tasks = n_tasks
#         self.n_features = n_features
#         self.n_classes = n_classes
#         self.residual = residual

#         n_layers = len(layer_sizes)

#         if not isinstance(weight_init_stddevs, Sequence):
#             weight_init_stddevs = [weight_init_stddevs] * n_layers
#         if not isinstance(bias_init_consts, Sequence):
#             bias_init_consts = [bias_init_consts] * n_layers
#         if not isinstance(dropouts, Sequence):
#             dropouts = [dropouts] * n_layers
#         if not isinstance(activation_fns, Sequence):
#             activation_fns = [activation_fns] * n_layers

#         # Store layers
#         self.layers = nn.ModuleList()
#         self.dropouts = nn.ModuleList()
#         self.activations = []

#         prev_size = n_features

#         for size, weight_stddev, bias_const, dropout, activation_fn in zip(
#             layer_sizes, weight_init_stddevs, bias_init_consts, dropouts, activation_fns
#         ):
#             dense = nn.Linear(prev_size, size)
#             # Initialize weights
#             nn.init.trunc_normal_(dense.weight, std=weight_stddev)
#             nn.init.constant_(dense.bias, bias_const)
#             self.layers.append(dense)
#             self.dropouts.append(nn.Dropout(dropout))
#             self.activations.append(activation_fn())
#             prev_size = size

#         # Final output layer
#         self.output_layer = nn.Linear(prev_size, n_tasks * n_classes)

#         # Store weight decay info
#         self.weight_decay_penalty = weight_decay_penalty
#         self.weight_decay_penalty_type = weight_decay_penalty_type

#     def forward(self, x):
#         prev = x
#         for dense, dropout, activation in zip(self.layers, self.dropouts, self.activations):
#             if self.residual and dense.in_features == dense.out_features:
#                 residual = prev
#                 out = activation(prev)
#                 out = dense(out)
#                 out = dropout(out)
#                 prev = residual + out
#             else:
#                 out = activation(prev)
#                 out = dense(out)
#                 out = dropout(out)
#                 prev = out

#         out = self.output_layer(prev)
#         logits = out.view(-1, self.n_tasks, self.n_classes)
#         output = F.softmax(logits, dim=-1)
#         return output, logits

#     def get_weight_decay_loss(self):
#         """Calculate weight decay regularization loss if needed."""
#         if self.weight_decay_penalty == 0.0:
#             return 0.0
#         penalty = 0.0
#         for param in self.parameters():
#             if self.weight_decay_penalty_type == "l2":
#                 penalty += torch.sum(param ** 2)
#             elif self.weight_decay_penalty_type == "l1":
#                 penalty += torch.sum(torch.abs(param))
#         return self.weight_decay_penalty * penalty

#     def default_generator(
#         self,
#         dataset,
#         batch_size: int,
#         epochs: int = 1,
#         deterministic: bool = True,
#         pad_batches: bool = True
#     ) -> Iterable[Tuple[List[torch.Tensor], List[torch.Tensor], List[torch.Tensor]]]:
#         """Mimic DeepChem TensorGraph default_generator behavior."""
#         for epoch in range(epochs):
#             for X_b, y_b, w_b, ids_b in dataset.iterbatches(
#                 batch_size=batch_size,
#                 deterministic=deterministic,
#                 pad_batches=pad_batches
#             ):
#                 X_b = torch.tensor(X_b).float()
#                 w_b = torch.tensor(w_b).float()
#                 if y_b is not None:
#                     y_b = torch.tensor(y_b).long()
#                     y_b_one_hot = F.one_hot(y_b.view(-1), num_classes=self.n_classes).float()
#                     y_b_one_hot = y_b_one_hot.view(-1, self.n_tasks, self.n_classes)
#                 else:
#                     y_b_one_hot = None

#                 yield ([X_b], [y_b_one_hot], [w_b])


#     def fit(self,
#             dataset: torch.utils.data.Dataset,
#             batch_size: int = 32,
#             nb_epoch: int = 10,
#             lr: float = 1e-3,
#             optimizer_cls=torch.optim.Adam,
#             device: str = 'cuda' if torch.cuda.is_available() else 'cpu') -> None:
#         """Simple fit method similar to Keras Model.fit()"""
#         self.to(device)
#         optimizer = optimizer_cls(self.parameters(), lr=lr)
#         loss_fn = nn.CrossEntropyLoss(reduction='none')  # We'll handle reduction manually
        
#         for epoch in range(nb_epoch):
#             self.train()
#             epoch_loss = 0.0
#             for (X_b_list, y_b_list, w_b_list) in self.default_generator(dataset, batch_size=batch_size, epochs=1):
#                 X_b, y_b, w_b = X_b_list[0].to(device), y_b_list[0].to(device), w_b_list[0].to(device)
#                 optimizer.zero_grad()
#                 preds, logits = self(X_b)

#                 logits_flat = logits.view(-1, self.n_classes)
#                 labels_flat = y_b.view(-1, self.n_classes)
#                 labels_idx = torch.argmax(labels_flat, dim=1)

#                 loss = loss_fn(logits_flat, labels_idx)
#                 # Apply task weights
#                 if w_b is not None:
#                     loss = loss * w_b.view(-1)

#                 loss = loss.mean()
#                 loss += self.get_weight_decay_loss()
#                 loss.backward()
#                 optimizer.step()

#                 epoch_loss += loss.item()

#             print(f"Epoch {epoch + 1}/{nb_epoch}, Loss: {epoch_loss:.4f}")

    
#     def evaluate(self, dataset, metrics, transformers=[], batch_size=100, device=None):
#         """
#         Mimics TensorGraph evaluate: returns a dict of metric scores.

#         Parameters
#         ----------
#         dataset: dc.data.Dataset
#             Dataset to evaluate on
#         metrics: list
#             List of dc.metrics.Metric objects
#         transformers: list
#             List of dc.trans.Transformer objects
#         batch_size: int
#             Minibatch size
#         device: torch.device
#             Device to run evaluation on
#         """
#         if device is None:
#             device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
#         self.eval()

#         y_true = []
#         y_pred = []

#         with torch.no_grad():
#             for X_b_list, y_b_list, w_b_list in self.default_generator(dataset, batch_size=batch_size, epochs=1, deterministic=True):
#                 X_b = X_b_list[0].to(device)
#                 y_b = y_b_list[0].to(device)
#                 output, logits = self(X_b)
#                 y_true.append(y_b.cpu().numpy())
#                 y_pred.append(output.cpu().numpy())

#         y_true = np.concatenate(y_true, axis=0)
#         y_pred = np.concatenate(y_pred, axis=0)

#         # Undo transformations (only if transformer has that method)
#         for transformer in transformers:
#             if hasattr(transformer, 'undo_labels'):
#                 y_true = transformer.undo_labels(y_true)

#         scores = {}
#         for metric in metrics:
#             scores[metric.name] = metric.compute_metric(y_true, y_pred)

#         return scores




In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from typing import List, Sequence, Union, Tuple, Iterable
import numpy as np

OneOrMany = Union[float, List[float]]

class MultitaskClassifier(nn.Module):
    """A fully connected network for multitask classification in PyTorch.
    
    This implementation closely mirrors the Keras implementation to ensure
    comparable performance, including pre-activation residual blocks.
    """
    
    def __init__(self,
                 n_tasks: int,
                 n_features: int,
                 layer_sizes: Sequence[int] = [1000],
                 weight_init_stddevs: OneOrMany = 0.02,
                 bias_init_consts: OneOrMany = 1.0,
                 weight_decay_penalty: float = 0.0,
                 weight_decay_penalty_type: str = "l2",
                 dropouts: OneOrMany = 0.5,
                 activation_fns: OneOrMany = nn.ReLU,
                 n_classes: int = 2,
                 residual: bool = False,
                 **kwargs) -> None:
        super(MultitaskClassifier, self).__init__()

        self.n_tasks = n_tasks
        self.n_features = n_features
        self.n_classes = n_classes
        self.residual = residual
        self.batch_size = kwargs.get('batch_size', 32)

        n_layers = len(layer_sizes)

        # Handle parameters that might be single values or sequences
        if not isinstance(weight_init_stddevs, Sequence) or isinstance(weight_init_stddevs, (str, bytes)):
            weight_init_stddevs = [weight_init_stddevs] * n_layers
        if not isinstance(bias_init_consts, Sequence) or isinstance(bias_init_consts, (str, bytes)):
            bias_init_consts = [bias_init_consts] * n_layers
        if not isinstance(dropouts, Sequence) or isinstance(dropouts, (str, bytes)):
            dropouts = [dropouts] * n_layers
        if not isinstance(activation_fns, Sequence) or isinstance(activation_fns, (type, str, bytes)):
            activation_fns = [activation_fns] * n_layers

        # Store weight decay info
        self.weight_decay_penalty = weight_decay_penalty
        self.weight_decay_penalty_type = weight_decay_penalty_type

        # Build the network - using ModuleList to store layers
        self.layers = nn.ModuleList()
        self.dropout_layers = nn.ModuleList()
        self.activation_layers = nn.ModuleList()
        
        prev_size = n_features
        
        # Build the dense layers matching the Keras implementation
        for i, (size, weight_stddev, bias_const, dropout, activation_fn) in enumerate(zip(
            layer_sizes, weight_init_stddevs, bias_init_consts, dropouts, activation_fns)):
            
            # Add activation for previous layer (except first layer)
            if i > 0:
                self.activation_layers.append(activation_fns[i-1]())
            else:
                # Placeholder for the first layer (won't be used)
                self.activation_layers.append(nn.Identity())
                
            # Add the dense layer
            dense = nn.Linear(prev_size, size)
            
            # Initialize weights with truncated normal
            with torch.no_grad():
                nn.init.trunc_normal_(dense.weight, std=weight_stddev)
                nn.init.constant_(dense.bias, bias_const)
                
            self.layers.append(dense)
            
            # Add dropout
            self.dropout_layers.append(nn.Dropout(dropout))
            
            prev_size = size
        
        # Add final activation
        self.final_activation = activation_fns[-1]()
        
        # Final output layer
        self.output_layer = nn.Linear(prev_size, n_tasks * n_classes)
        nn.init.trunc_normal_(self.output_layer.weight, std=weight_init_stddevs[-1])
        nn.init.constant_(self.output_layer.bias, bias_init_consts[-1])

    def forward(self, x):
        prev_layer = x
        
        # Process the dense layers
        for i, (dense, dropout) in enumerate(zip(self.layers, self.dropout_layers)):
            # Get the activation for this layer (pre-activation pattern)
            activation = self.activation_layers[i]
            
            if i > 0:  # First layer doesn't have pre-activation
                layer = activation(prev_layer)
            else:
                layer = prev_layer
                
            layer = dense(layer)
            
            if dropout.p > 0.0:
                layer = dropout(layer)
                
            # Apply residual connection if applicable
            if self.residual and dense.in_features == dense.out_features:
                prev_layer = prev_layer + layer
            else:
                prev_layer = layer
        
        # Apply final activation
        neural_fingerprint = self.final_activation(prev_layer)
        
        # Output layer
        logits = self.output_layer(neural_fingerprint)
        logits = logits.reshape(-1, self.n_tasks, self.n_classes)
        output = F.softmax(logits, dim=2)
        
        return output, logits
    
    def get_neural_fingerprint(self, x):
        """Extract the neural fingerprint for the input."""
        with torch.no_grad():
            prev_layer = x
            
            # Process the dense layers
            for i, (dense, dropout) in enumerate(zip(self.layers, self.dropout_layers)):
                # Get the activation for this layer
                activation = self.activation_layers[i]
                
                if i > 0:  # First layer doesn't have pre-activation
                    layer = activation(prev_layer)
                else:
                    layer = prev_layer
                    
                layer = dense(layer)
                
                if dropout.p > 0.0:
                    layer = dropout(layer)
                    
                # Apply residual connection if applicable
                if self.residual and dense.in_features == dense.out_features:
                    prev_layer = prev_layer + layer
                else:
                    prev_layer = layer
            
            # Apply final activation
            neural_fingerprint = self.final_activation(prev_layer)
            
        return neural_fingerprint

    def get_weight_decay_loss(self):
        """Calculate weight decay regularization loss."""
        if self.weight_decay_penalty == 0.0:
            return 0.0
            
        penalty = 0.0
        for param in self.parameters():
            if self.weight_decay_penalty_type == "l2":
                penalty += torch.sum(param ** 2)
            elif self.weight_decay_penalty_type == "l1":
                penalty += torch.sum(torch.abs(param))
                
        return self.weight_decay_penalty * penalty

    def default_generator(
        self,
        dataset,
        batch_size: int = None,
        epochs: int = 1,
        deterministic: bool = True,
        pad_batches: bool = True
    ) -> Iterable[Tuple[List[torch.Tensor], List[torch.Tensor], List[torch.Tensor]]]:
        """Generate batches from dataset, similar to DeepChem's default_generator."""
        if batch_size is None:
            batch_size = self.batch_size
            
        for epoch in range(epochs):
            for X_b, y_b, w_b, ids_b in dataset.iterbatches(
                batch_size=batch_size,
                deterministic=deterministic,
                pad_batches=pad_batches
            ):
                X_b = torch.tensor(X_b, dtype=torch.float32)
                w_b = torch.tensor(w_b, dtype=torch.float32)
                
                if y_b is not None:
                    # Handle one-hot encoding consistent with Keras version
                    y_b = torch.tensor(y_b, dtype=torch.long)
                    y_flat = y_b.flatten()
                    y_one_hot = F.one_hot(y_flat, num_classes=self.n_classes).float()
                    y_b_one_hot = y_one_hot.reshape(-1, self.n_tasks, self.n_classes)
                else:
                    y_b_one_hot = None

                yield ([X_b], [y_b_one_hot], [w_b])

    def fit(self,
            dataset,
            nb_epoch: int = 10,
            batch_size: int = None,
            lr: float = 1e-3,
            optimizer_cls=torch.optim.Adam,
            device: str = None) -> None:
        """Train the model on the given dataset.
        
        Parameters
        ----------
        dataset: Dataset object that supports iterbatches
        nb_epoch: Number of training epochs
        batch_size: Batch size for training
        lr: Learning rate
        optimizer_cls: PyTorch optimizer class
        device: Device to use for training
        """
        if device is None:
            device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        if batch_size is None:
            batch_size = self.batch_size
            
        self.to(device)
        optimizer = optimizer_cls(self.parameters(), lr=lr)
        
        for epoch in range(nb_epoch):
            self.train()
            epoch_loss = 0.0
            n_batches = 0
            
            for (X_b_list, y_b_list, w_b_list) in self.default_generator(
                dataset, batch_size=batch_size, epochs=1, deterministic=True):
                
                X_b = X_b_list[0].to(device)
                y_b = y_b_list[0].to(device) if y_b_list[0] is not None else None
                w_b = w_b_list[0].to(device)
                
                optimizer.zero_grad()
                
                # Forward pass
                outputs, logits = self(X_b)
                
                # Calculate loss - using cross entropy loss
                # First reshape to [batch_size * n_tasks, n_classes]
                logits_flat = logits.reshape(-1, self.n_classes)
                y_flat = y_b.reshape(-1, self.n_classes)
                
                # Get class indices from one-hot
                y_indices = torch.argmax(y_flat, dim=1)
                
                # Apply task weights
                loss_per_example = F.cross_entropy(logits_flat, y_indices, reduction='none')
                
                # Reshape loss to match the shape of w_b
                loss_per_task = loss_per_example.reshape(-1, self.n_tasks)
                weighted_loss = loss_per_task * w_b
                
                # Average over batch and tasks
                loss = weighted_loss.sum() / (w_b.sum() + 1e-10)
                
                # Add weight decay
                loss += self.get_weight_decay_loss()
                
                # Backward pass and optimize
                loss.backward()
                optimizer.step()
                
                epoch_loss += loss.item()
                n_batches += 1
            
            print(f"Epoch {epoch+1}/{nb_epoch}, Loss: {epoch_loss/n_batches:.4f}")
    
    def predict(self, dataset, batch_size=None, transformers=[], device=None):
        """
        Make predictions on the dataset.
        
        Parameters
        ----------
        dataset: Dataset
            Dataset to make prediction on
        batch_size: int
            Batch size for prediction
        transformers: list
            List of transformer objects
        device: torch.device
            Device to run prediction on
        """
        if device is None:
            device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        if batch_size is None:
            batch_size = self.batch_size
            
        self.eval()
        self.to(device)
        
        y_preds = []
        
        with torch.no_grad():
            for X_b_list, _, _, _ in dataset.iterbatches(
                batch_size=batch_size, deterministic=True, pad_batches=True):
                
                X_b = torch.tensor(X_b_list, dtype=torch.float32).to(device)
                outputs, _ = self(X_b)
                y_preds.append(outputs.cpu().numpy())
        
        y_pred = np.concatenate(y_preds, axis=0)
        
        # Undo transformations if needed
        for transformer in transformers:
            if hasattr(transformer, 'undo_transform'):
                y_pred = transformer.undo_transform(y_pred)
                
        return y_pred

    def evaluate(self, dataset, metrics, transformers=[], batch_size=None, device=None):
        """
        Evaluate model performance on a dataset.
        
        Parameters
        ----------
        dataset: Dataset
            Dataset to evaluate on
        metrics: list
            List of metric objects
        transformers: list
            List of transformer objects
        batch_size: int
            Batch size for evaluation
        device: torch.device
            Device to run evaluation on
        """
        if device is None:
            device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        if batch_size is None:
            batch_size = self.batch_size
            
        self.eval()
        self.to(device)
        
        y_true = []
        y_pred = []
        
        with torch.no_grad():
            for X_b_list, y_b_list, w_b_list in self.default_generator(
                dataset, batch_size=batch_size, epochs=1, deterministic=True):
                
                X_b = X_b_list[0].to(device)
                y_b = y_b_list[0]
                
                if y_b is not None:
                    y_true.append(y_b.numpy())
                    
                    outputs, _ = self(X_b)
                    y_pred.append(outputs.cpu().numpy())
        
        if len(y_true) == 0:
            return {}
            
        y_true = np.concatenate(y_true, axis=0)
        y_pred = np.concatenate(y_pred, axis=0)
        
        # Undo transformations if needed
        for transformer in transformers:
            if hasattr(transformer, 'undo_transform'):
                if y_true.shape == y_pred.shape:
                    y_true = transformer.undo_transform(y_true)
                y_pred = transformer.undo_transform(y_pred)
        
        # Calculate metrics
        results = {}
        for metric in metrics:
            results[metric.name] = metric.compute_metric(y_true, y_pred)
            
        return results

For classification, we'll use a simple fully connected network with one hidden layer.

In [ ]:
classifier = MultitaskClassifier(
    n_tasks=len(tasks),
    n_features=256,
    layer_sizes=[512]
)

# Train the classifier
classifier.fit(train_embeddings_dataset, nb_epoch=10)

Epoch 1/10, Loss: 0.6008
Epoch 2/10, Loss: 0.5847
Epoch 3/10, Loss: 0.5521
Epoch 4/10, Loss: 0.5119
Epoch 5/10, Loss: 0.4954
Epoch 6/10, Loss: 0.4598
Epoch 7/10, Loss: 0.4562
Epoch 8/10, Loss: 0.4361
Epoch 9/10, Loss: 0.4906
Epoch 10/10, Loss: 0.4342


In [ ]:
import deepchem.models
print(deepchem.models.MultitaskClassifier.__bases__)
print(deepchem.__file__)

(<class 'deepchem.models.keras_model.KerasModel'>,)
/home/jantine/miniconda3/envs/deepchem/lib/python3.12/site-packages/deepchem/__init__.py


In [ ]:
# class MultitaskClassifier(KerasModel):
#   """A fully connected network for multitask classification.

#   This class provides lots of options for customizing aspects of the model: the
#   number and widths of layers, the activation functions, regularization methods,
#   etc.

#   It optionally can compose the model from pre-activation residual blocks, as
#   described in https://arxiv.org/abs/1603.05027, rather than a simple stack of
#   dense layers.  This often leads to easier training, especially when using a
#   large number of layers.  Note that residual blocks can only be used when
#   successive layers have the same width.  Wherever the layer width changes, a
#   simple dense layer will be used even if residual=True.
#   """

#   def __init__(self,
#                n_tasks: int,
#                n_features: int,
#                layer_sizes: Sequence[int] = [1000],
#                weight_init_stddevs: OneOrMany[float] = 0.02,
#                bias_init_consts: OneOrMany[float] = 1.0,
#                weight_decay_penalty: float = 0.0,
#                weight_decay_penalty_type: str = "l2",
#                dropouts: OneOrMany[float] = 0.5,
#                activation_fns: OneOrMany[KerasActivationFn] = tf.nn.relu,
#                n_classes: int = 2,
#                residual: bool = False,
#                **kwargs) -> None:
#     """Create a MultitaskClassifier.

#     In addition to the following arguments, this class also accepts
#     all the keyword arguments from TensorGraph.

#     Parameters
#     ----------
#     n_tasks: int
#       number of tasks
#     n_features: int
#       number of features
#     layer_sizes: list
#       the size of each dense layer in the network.  The length of
#       this list determines the number of layers.
#     weight_init_stddevs: list or float
#       the standard deviation of the distribution to use for weight
#       initialization of each layer.  The length of this list should
#       equal len(layer_sizes).  Alternatively this may be a single
#       value instead of a list, in which case the same value is used
#       for every layer.
#     bias_init_consts: list or float
#       the value to initialize the biases in each layer to.  The
#       length of this list should equal len(layer_sizes).
#       Alternatively this may be a single value instead of a list, in
#       which case the same value is used for every layer.
#     weight_decay_penalty: float
#       the magnitude of the weight decay penalty to use
#     weight_decay_penalty_type: str
#       the type of penalty to use for weight decay, either 'l1' or 'l2'
#     dropouts: list or float
#       the dropout probablity to use for each layer.  The length of this list should equal len(layer_sizes).
#       Alternatively this may be a single value instead of a list, in which case the same value is used for every layer.
#     activation_fns: list or object
#       the Tensorflow activation function to apply to each layer.  The length of this list should equal
#       len(layer_sizes).  Alternatively this may be a single value instead of a list, in which case the
#       same value is used for every layer.
#     n_classes: int
#       the number of classes
#     residual: bool
#       if True, the model will be composed of pre-activation residual blocks instead
#       of a simple stack of dense layers.
#     """
#     self.n_tasks = n_tasks
#     self.n_features = n_features
#     self.n_classes = n_classes
#     n_layers = len(layer_sizes)
#     if not isinstance(weight_init_stddevs, SequenceCollection):
#       weight_init_stddevs = [weight_init_stddevs] * n_layers
#     if not isinstance(bias_init_consts, SequenceCollection):
#       bias_init_consts = [bias_init_consts] * n_layers
#     if not isinstance(dropouts, SequenceCollection):
#       dropouts = [dropouts] * n_layers
#     if not isinstance(activation_fns, SequenceCollection):
#       activation_fns = [activation_fns] * n_layers
#     if weight_decay_penalty != 0.0:
#       if weight_decay_penalty_type == 'l1':
#         regularizer = tf.keras.regularizers.l1(weight_decay_penalty)
#       else:
#         regularizer = tf.keras.regularizers.l2(weight_decay_penalty)
#     else:
#       regularizer = None

#     # Add the input features.

#     mol_features = Input(shape=(n_features,))
#     prev_layer = mol_features
#     prev_size = n_features
#     next_activation = None

#     # Add the dense layers

#     for size, weight_stddev, bias_const, dropout, activation_fn in zip(
#         layer_sizes, weight_init_stddevs, bias_init_consts, dropouts,
#         activation_fns):
#       layer = prev_layer
#       if next_activation is not None:
#         layer = Activation(next_activation)(layer)
#       layer = Dense(
#           size,
#           kernel_initializer=tf.keras.initializers.TruncatedNormal(
#               stddev=weight_stddev),
#           bias_initializer=tf.constant_initializer(value=bias_const),
#           kernel_regularizer=regularizer)(layer)
#       if dropout > 0.0:
#         layer = Dropout(rate=dropout)(layer)
#       if residual and prev_size == size:
#         prev_layer = Lambda(lambda x: x[0] + x[1])([prev_layer, layer])
#       else:
#         prev_layer = layer
#       prev_size = size
#       next_activation = activation_fn
#     if next_activation is not None:
#       prev_layer = Activation(activation_fn)(prev_layer)
#     self.neural_fingerprint = prev_layer
#     logits = Reshape((n_tasks,
#                       n_classes))(Dense(n_tasks * n_classes)(prev_layer))
#     output = Softmax()(logits)
#     model = tf.keras.Model(inputs=mol_features, outputs=[output, logits])
#     super(MultitaskClassifier, self).__init__(
#         model,
#         dc.models.losses.SoftmaxCrossEntropy(),
#         output_types=['prediction', 'loss'],
#         **kwargs)

#   def default_generator(
#       self,
#       dataset: dc.data.Dataset,
#       epochs: int = 1,
#       mode: str = 'fit',
#       deterministic: bool = True,
#       pad_batches: bool = True) -> Iterable[Tuple[List, List, List]]:
#     for epoch in range(epochs):
#       for (X_b, y_b, w_b, ids_b) in dataset.iterbatches(
#           batch_size=self.batch_size,
#           deterministic=deterministic,
#           pad_batches=pad_batches):
#         if y_b is not None:
#           y_b = to_one_hot(y_b.flatten(), self.n_classes).reshape(
#               -1, self.n_tasks, self.n_classes)
#         yield ([X_b], [y_b], [w_b])

Find out how well it worked.  Compute the ROC AUC for the training and validation datasets.

In [ ]:
metric = dc.metrics.Metric(dc.metrics.roc_auc_score, np.mean, mode="classification")
train_score = classifier.evaluate(train_embeddings_dataset, [metric], transformers)
valid_score = classifier.evaluate(valid_embeddings_dataset, [metric], transformers)
print('Training set ROC AUC:', train_score)
print('Validation set ROC AUC:', valid_score)

Training set ROC AUC: {'mean-roc_auc_score': np.float64(0.5089696806891291)}
Validation set ROC AUC: {'mean-roc_auc_score': np.float64(0.49178838759073346)}


# Congratulations! Time to join the Community!

Congratulations on completing this tutorial notebook! If you enjoyed working through the tutorial, and want to continue working with DeepChem, we encourage you to finish the rest of the tutorials in this series. You can also help the DeepChem community in the following ways:

## Star DeepChem on [GitHub](https://github.com/deepchem/deepchem)
This helps build awareness of the DeepChem project and the tools for open source drug discovery that we're trying to build.

## Join the DeepChem Gitter
The DeepChem [Gitter](https://gitter.im/deepchem/Lobby) hosts a number of scientists, developers, and enthusiasts interested in deep learning for the life sciences. Join the conversation!